# Table of Contents
## CONFIG — update before each run
## 1. 1-min bars → GoodOldGoodOld.csv
## 2. 1-sec bars (09:30–10:00) → NewGoodOldGoodOld.csv

In [ ]:
import sys
print(sys.executable)

## CONFIG — update before each run

In [ ]:
from datetime import datetime

# ── Update these values each quarter / run ──────────────────────────────────
CONTRACT_MONTH   = '202506'              # Current ES contract expiry (YYYYMM)

# Section 1 — 1-min bars
MINUTES_END_DATE = datetime(2026, 4, 1)  # Last date of the desired fetch window
MINUTES_DURATION = '1 M'                 # IBKR duration string (e.g. '1 M', '2 W')

# Section 2 — 1-sec bars: iterates every weekday in [SECS_START_DATE, SECS_END_DATE]
SECS_START_DATE  = datetime(2026, 3, 1)  # First day (inclusive)
SECS_END_DATE    = datetime(2026, 4, 1)  # Last day (inclusive)
# ───────────────────────────────────────────────────────────────────────

print(f'Contract month : {CONTRACT_MONTH}')
print(f'1-min end date : {MINUTES_END_DATE.date()}  duration: {MINUTES_DURATION}')
print(f'1-sec range    : {SECS_START_DATE.date()} → {SECS_END_DATE.date()}')

## 1. 1-min bars → GoodOldGoodOld.csv
Fetches OHLCV 1-minute bars ending on `MINUTES_END_DATE` and appends to `GoodOldGoodOld.csv`.

In [ ]:
import asyncio
import pandas as pd
from ib_insync import IB, Future
import os
from datetime import datetime

# Requires: CONFIG cell must be run first.
# Requires: IBKR TWS or IB Gateway running on localhost:7496.

ib = IB()

if ib.isConnected():
    print('Disconnecting previous connection...')
    ib.disconnect()

try:
    await ib.connectAsync('127.0.0.1', 7496, clientId=1)
    print('Connected to IBKR.')
except Exception as e:
    print(f'Failed to connect to IBKR: {e}')
    raise

contract = Future(symbol='ES', lastTradeDateOrContractMonth=CONTRACT_MONTH, exchange='CME', includeExpired=True)
await ib.qualifyContractsAsync(contract)

existing_file = 'GoodOldGoodOld.csv'

await asyncio.sleep(1)
try:
    bars = await ib.reqHistoricalDataAsync(
        contract,
        endDateTime=MINUTES_END_DATE,
        durationStr=MINUTES_DURATION,
        barSizeSetting='1 min',
        whatToShow='TRADES',
        useRTH=False
    )
    if not bars:
        ib.disconnect()
        raise RuntimeError('IBKR returned no data — check subscription, pacing limits, or date range.')
    print(f'Fetched {len(bars)} raw bars.')
except Exception as e:
    print(f'Failed to fetch historical data: {e}')
    ib.disconnect()
    raise

new_data = pd.DataFrame([b.dict() for b in bars])

if 'date' not in new_data.columns:
    print('Error: date column not found.')
    print(new_data.head())
    ib.disconnect()
    raise SystemExit

new_data['date'] = pd.to_datetime(new_data['date'])
if new_data['date'].dt.tz is not None:
    new_data['date'] = new_data['date'].dt.tz_convert('US/Eastern')
else:
    new_data['date'] = new_data['date'].dt.tz_localize('UTC').dt.tz_convert('US/Eastern')

new_data = new_data[new_data['date'].dt.weekday < 5]  # Monday–Friday only
new_data['time'] = new_data['date'].dt.strftime('%H:%M:%S')

specific_times = [
    '09:30:00', '09:31:00', '09:32:00', '09:33:00',
    '09:34:00', '09:35:00', '09:36:00', '09:38:00', '09:40:00', '09:43:00',
    '09:45:00', '09:46:00', '09:50:00', '09:51:00', '09:55:00', '09:57:00',
    '10:00:00', '10:04:00', '10:13:00', '10:15:00', '10:25:00', '10:30:00',
    '10:40:00', '10:45:00', '10:59:00', '11:00:00', '11:15:00', '11:23:00',
    '11:30:00', '11:54:00', '12:00:00', '12:15:00', '12:30:00', '12:32:00',
    '12:45:00', '13:00:00', '13:15:00', '13:30:00', '13:45:00', '13:53:00',
    '14:00:00', '14:15:00', '14:25:00', '14:30:00', '14:45:00', '15:00:00',
    '15:15:00', '15:30:00', '15:45:00', '15:47:00', '16:00:00', '16:15:00'
]

filtered_data = new_data[new_data['time'].isin(specific_times)]
print(f'Rows after time filter: {len(filtered_data)}')

if os.path.exists(existing_file):
    existing_data = pd.read_csv(existing_file)
    existing_data['date'] = pd.to_datetime(existing_data['date'])
    combined_data = pd.concat([existing_data, filtered_data]).drop_duplicates(subset='date', keep='last')
else:
    combined_data = filtered_data

combined_data.sort_values(by='date', inplace=True)
combined_data.to_csv(existing_file, index=False)
print(f'Saved {existing_file!r}. Total rows: {len(combined_data)}')

if ib.isConnected():
    ib.disconnect()
    print('Disconnected from IBKR.')

## 2. 1-sec bars (09:30–10:00) → NewGoodOldGoodOld.csv
Iterates every weekday in `[SECS_START_DATE, SECS_END_DATE]`, fetching 30 min of 1-second bars ending at 10:00 ET per day.

In [ ]:
import asyncio
import pandas as pd
from ib_insync import IB, Future
import os
from datetime import datetime, timedelta

# Requires: CONFIG cell must be run first.
# Requires: IBKR TWS or IB Gateway running on localhost:7496.
# Uses reqHistoricalTicksAsync (Time & Sales backend) instead of HMDS —
# allows fetching tick data much further back than 1-second bars.
# Note: 10-second sleep between days to avoid IBKR pacing violations.
#
# IBKR reqHistoricalTicks:
#   - Accepts ONE of startDateTime or endDateTime, not both.
#   - Most reliable format is UTC with dash: 'YYYYMMDD-HH:MM:SS'
#   - 'YYYYMMDD HH:MM:SS US/Eastern' causes Error 10314 in some TWS versions.
#   - We convert 09:30:00 US/Eastern → UTC via pandas (handles DST automatically).

def iterate_weekdays(start_date, end_date):
    current_date = start_date
    while current_date <= end_date:
        if current_date.weekday() < 5:  # Monday–Friday only
            yield current_date
        current_date += timedelta(days=1)

ib = IB()

if ib.isConnected():
    print('Disconnecting previous connection...')
    ib.disconnect()

try:
    await ib.connectAsync('127.0.0.1', 7496, clientId=1)
    print('Connected to IBKR.')
except Exception as e:
    print(f'Failed to connect to IBKR: {e}')
    raise

contract = Future(symbol='ES', lastTradeDateOrContractMonth=CONTRACT_MONTH, exchange='CME', includeExpired=True)
await ib.qualifyContractsAsync(contract)

existing_file = 'NewGoodOldGoodOld.csv'

# Sub-second timestamps to capture within the first 55 seconds of the open.
specific_times = [
    '09:30:01', '09:30:02', '09:30:03', '09:30:05', '09:30:08', '09:30:10',
    '09:30:11', '09:30:13', '09:30:15', '09:30:20', '09:30:21', '09:30:25',
    '09:30:30', '09:30:34', '09:30:35', '09:30:40', '09:30:45', '09:30:50',
    '09:30:55'
]

for day in iterate_weekdays(SECS_START_DATE, SECS_END_DATE):
    # Convert 09:30:00 US/Eastern → UTC, format as 'YYYYMMDD-HH:MM:SS' (IBKR UTC notation).
    # pandas handles DST automatically (EST=UTC-5 vs EDT=UTC-4).
    start_ts = pd.Timestamp(f'{day.strftime("%Y-%m-%d")} 09:30:00', tz='US/Eastern')
    start_dt_str = start_ts.tz_convert('UTC').strftime('%Y%m%d-%H:%M:%S')
    print(f'Fetching {day.strftime("%Y-%m-%d")} (startDateTime={start_dt_str})...')

    try:
        ticks = await ib.reqHistoricalTicksAsync(
            contract,
            startDateTime=start_dt_str,
            endDateTime='',
            numberOfTicks=1000,
            whatToShow='MIDPOINT',
            useRth=True,
            ignoreSize=False,
        )
        await asyncio.sleep(10)  # IBKR pacing: wait between requests
        print(f'  Fetched {len(ticks)} raw ticks.')
    except Exception as e:
        print(f'  Failed to fetch {day.strftime("%Y-%m-%d")}: {e}')
        ib.disconnect()
        raise

    if not ticks:
        print(f'  No ticks returned for {day.strftime("%Y-%m-%d")} — skipping.')
        continue

    # Build a DataFrame of ticks and localise to US/Eastern.
    tick_df = pd.DataFrame([{'date': t.time, 'price': t.price} for t in ticks])
    tick_df['date'] = pd.to_datetime(tick_df['date'])
    if tick_df['date'].dt.tz is not None:
        tick_df['date'] = tick_df['date'].dt.tz_convert('US/Eastern')
    else:
        tick_df['date'] = tick_df['date'].dt.tz_localize('UTC').dt.tz_convert('US/Eastern')
    tick_df = tick_df.sort_values('date').reset_index(drop=True)

    # For each target second, find the last tick at-or-before that second.
    rows = []
    for t_str in specific_times:
        target_dt = pd.Timestamp(f'{day.strftime("%Y-%m-%d")} {t_str}', tz='US/Eastern')
        candidates = tick_df[tick_df['date'] <= target_dt]
        if candidates.empty:
            continue
        last_tick = candidates.iloc[-1]
        rows.append({
            'date':  target_dt,
            'open':  last_tick['price'],
            'high':  last_tick['price'],
            'low':   last_tick['price'],
            'close': last_tick['price'],
            'volume': 0,
            'time':  t_str,
        })

    if not rows:
        print(f'  No matching timestamps found for {day.strftime("%Y-%m-%d")} — skipping.')
        continue

    filtered_data = pd.DataFrame(rows)
    print(f'  Matched {len(filtered_data)} target timestamps.')

    if os.path.exists(existing_file):
        existing_data = pd.read_csv(existing_file)
        existing_data['date'] = pd.to_datetime(existing_data['date'])
        combined_data = pd.concat([existing_data, filtered_data]).drop_duplicates(subset='date', keep='last')
    else:
        combined_data = filtered_data

    combined_data.sort_values(by='date', inplace=True)
    combined_data.to_csv(existing_file, index=False)
    print(f'  Saved. Total rows in file: {len(combined_data)}')

if ib.isConnected():
    ib.disconnect()
    print('Disconnected from IBKR.')